Python / ML + GenAI – Combining classic features, sentiment, and embeddings (intermediate)
You have a small set of app store reviews with metadata. Save as day7_app_reviews.csv:

text
review_id,user_tenure_days,sessions_last_7d,platform,country,review_text,stars
1,30,5,Android,IN,"App is fast and easy to use, love the new update!",5
2,200,2,iOS,US,"Works fine but drains battery quickly.",3
3,10,1,Android,IN,"Crashes every time I open it, very frustrating.",1
4,365,8,Web,UK,"Reliable for my daily work, a few minor bugs.",4
5,90,3,iOS,IN,"Good features but the interface is confusing.",3
6,45,6,Android,US,"Amazing experience, support team was very helpful.",5
7,150,4,Web,UK,"Slow on older devices, but overall acceptable.",3
8,7,2,Android,IN,"Terrible onboarding, I uninstalled after a day.",1
9,120,7,iOS,US,"Solid performance, much better than before.",4
10,300,5,Web,IN,"Great for collaboration, team loves it.",5
Goal: Practice thinking about feature types (numeric, categorical, GenAI‑derived) and how they combine.

Tasks:

Load data and basic EDA:

Summaries of numeric features and distribution of stars.

Classic features:
Create review_length (characters or tokens).

Build a baseline regression model (e.g., RandomForestRegressor or Ridge) using:
Numeric: user_tenure_days, sessions_last_7d, review_length.
Categorical: platform, country (OneHotEncoder).

Train/test split and compute RMSE and R2 score
 .

GenAI layer 1 – sentiment with a Hugging Face pipeline:
Use pipeline("sentiment-analysis") to get sentiment_label and sentiment_score for each review_text.
Map sentiment_label to a numeric feature (e.g., POSITIVE=1, NEGATIVE=0) and keep sentiment_score as numeric.
Re‑train model including sentiment features:
Compare performance vs baseline.
Reflect: did adding sentiment help on this tiny dataset?

GenAI layer 2 – embeddings (concept):
Use a sentence embedding model (e.g., all-MiniLM-L6-v2) to encode review_text.

Do either:
Train a simple model just on embeddings, or
Keep it conceptual: inspect cosine similarities between some pairs of reviews (e.g., two 1‑star, two 5‑star) to see if similar‑meaning reviews are close in embedding space.

Concept reflection (write as markdown or here):
When would you rely on:
pure tabular features only,
tabular + sentiment,
full embeddings (or an LLM) for rating/feedback modeling in a real product?

In [1]:
import pandas as pd
from io import StringIO

data = """review_id,user_tenure_days,sessions_last_7d,platform,country,review_text,stars
1,30,5,Android,IN,"App is fast and easy to use, love the new update!",5
2,200,2,iOS,US,"Works fine but drains battery quickly.",3
3,10,1,Android,IN,"Crashes every time I open it, very frustrating.",1
4,365,8,Web,UK,"Reliable for my daily work, a few minor bugs.",4
5,90,3,iOS,IN,"Good features but the interface is confusing.",3
6,45,6,Android,US,"Amazing experience, support team was very helpful.",5
7,150,4,Web,UK,"Slow on older devices, but overall acceptable.",3
8,7,2,Android,IN,"Terrible onboarding, I uninstalled after a day.",1
9,120,7,iOS,US,"Solid performance, much better than before.",4
10,300,5,Web,IN,"Great for collaboration, team loves it.",5"""

df = pd.read_csv(StringIO(data))

df

,review_id,user_tenure_days,sessions_last_7d,platform,country,review_text,stars
0,1,30,5,Android,IN,"App is fast and easy to use, love the new update!",5
1,2,200,2,iOS,US,Works fine but drains battery quickly.,3
2,3,10,1,Android,IN,"Crashes every time I open it, very frustrating.",1
3,4,365,8,Web,UK,"Reliable for my daily work, a few minor bugs.",4
4,5,90,3,iOS,IN,Good features but the interface is confusing.,3
5,6,45,6,Android,US,"Amazing experience, support team was very help...",5
6,7,150,4,Web,UK,"Slow on older devices, but overall acceptable.",3
7,8,7,2,Android,IN,"Terrible onboarding, I uninstalled after a day.",1
8,9,120,7,iOS,US,"Solid performance, much better than before.",4
9,10,300,5,Web,IN,"Great for collaboration, team loves it.",5


In [2]:
df.describe()

,review_id,user_tenure_days,sessions_last_7d,stars
count,10.00000,10.000000,10.000000,10.000000
mean,5.50000,131.700000,4.300000,3.400000
std,3.02765,123.760566,2.311805,1.505545
min,1.00000,7.000000,1.000000,1.000000
25%,3.25000,33.750000,2.250000,3.000000
50%,5.50000,105.000000,4.500000,3.500000
75%,7.75000,187.500000,5.750000,4.750000
max,10.00000,365.000000,8.000000,5.000000


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   review_id         10 non-null     int64 
 1   user_tenure_days  10 non-null     int64 
 2   sessions_last_7d  10 non-null     int64 
 3   platform          10 non-null     object
 4   country           10 non-null     object
 5   review_text       10 non-null     object
 6   stars             10 non-null     int64 
dtypes: int64(4), object(3)
memory usage: 692.0+ bytes


In [5]:
df.isnull().sum()

review_id           0
user_tenure_days    0
sessions_last_7d    0
platform            0
country             0
review_text         0
stars               0
dtype: int64

In [8]:
df['review_length'] = df['review_text'].str.len()

df


,review_id,user_tenure_days,sessions_last_7d,platform,country,review_text,stars,review_length
0,1,30,5,Android,IN,"App is fast and easy to use, love the new update!",5,49
1,2,200,2,iOS,US,Works fine but drains battery quickly.,3,38
2,3,10,1,Android,IN,"Crashes every time I open it, very frustrating.",1,47
3,4,365,8,Web,UK,"Reliable for my daily work, a few minor bugs.",4,45
4,5,90,3,iOS,IN,Good features but the interface is confusing.,3,45
5,6,45,6,Android,US,"Amazing experience, support team was very help...",5,50
6,7,150,4,Web,UK,"Slow on older devices, but overall acceptable.",3,46
7,8,7,2,Android,IN,"Terrible onboarding, I uninstalled after a day.",1,47
8,9,120,7,iOS,US,"Solid performance, much better than before.",4,43
9,10,300,5,Web,IN,"Great for collaboration, team loves it.",5,39


In [11]:
X = df.drop('stars',axis = 1)
y = df['stars']


In [12]:
X

,review_id,user_tenure_days,sessions_last_7d,platform,country,review_text,review_length
0,1,30,5,Android,IN,"App is fast and easy to use, love the new update!",49
1,2,200,2,iOS,US,Works fine but drains battery quickly.,38
2,3,10,1,Android,IN,"Crashes every time I open it, very frustrating.",47
3,4,365,8,Web,UK,"Reliable for my daily work, a few minor bugs.",45
4,5,90,3,iOS,IN,Good features but the interface is confusing.,45
5,6,45,6,Android,US,"Amazing experience, support team was very help...",50
6,7,150,4,Web,UK,"Slow on older devices, but overall acceptable.",46
7,8,7,2,Android,IN,"Terrible onboarding, I uninstalled after a day.",47
8,9,120,7,iOS,US,"Solid performance, much better than before.",43
9,10,300,5,Web,IN,"Great for collaboration, team loves it.",39


In [13]:
y

0    5
1    3
2    1
3    4
4    3
5    5
6    3
7    1
8    4
9    5
Name: stars, dtype: int64

In [21]:
Numeric_features = ["user_tenure_days", "sessions_last_7d", "review_length"]
Categorical_features = ["platform", "country"]

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

preprocessing = ColumnTransformer(transformers = [("Num_scaling", StandardScaler(),Numeric_features),
                 ("Cat_scaling", OneHotEncoder(handle_unknown = 'ignore'),Categorical_features)])


from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(X,y,random_state = 42,test_size = 0.3)


from sklearn.ensemble import RandomForestRegressor
steps = [("preprossing",preprocessing),
         ("RandomForest", RandomForestRegressor(n_estimators= 300,random_state = 42))]

pipe1 = Pipeline(steps = steps, verbose = True)

In [22]:
pipe1

Pipeline(steps=[('preprossing',
                 ColumnTransformer(transformers=[('Num_scaling',
                                                  StandardScaler(),
                                                  ['user_tenure_days',
                                                   'sessions_last_7d',
                                                   'review_length']),
                                                 ('Cat_scaling',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['platform', 'country'])])),
                ('RandomForest',
                 RandomForestRegressor(n_estimators=300, random_state=42))],
         verbose=True)

In [23]:
pipe1.fit(X_train,y_train)

Pipeline(steps=[('preprossing',
                 ColumnTransformer(transformers=[('Num_scaling',
                                                  StandardScaler(),
                                                  ['user_tenure_days',
                                                   'sessions_last_7d',
                                                   'review_length']),
                                                 ('Cat_scaling',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['platform', 'country'])])),
                ('RandomForest',
                 RandomForestRegressor(n_estimators=300, random_state=42))],
         verbose=True)

In [25]:
y_pred = pipe1.predict(X_test)
y_pred

array([4.01      , 2.59      , 3.91666667])

In [26]:
from sklearn.metrics import root_mean_squared_error, mean_squared_error, r2_score

rmse = root_mean_squared_error(y_test,y_pred)
print("RMSE score is : {} ".format(rmse))
r2_Score = r2_score(y_test, y_pred)
print("r2 score is : {} ".format(r2_Score))

RMSE score is : 0.6687827527458902 
r2 score is : 0.32909444444444425 


In [27]:
from transformers import pipeline
import pandas as pd

sentiment_pipeline = pipeline("sentiment-analysis")

sentiments = sentiment_pipeline(list(df['review_text']))

sentiments

[{'label': 'POSITIVE', 'score': 0.9993120431900024},
 {'label': 'NEGATIVE', 'score': 0.994539201259613},
 {'label': 'NEGATIVE', 'score': 0.9995972514152527},
 {'label': 'POSITIVE', 'score': 0.9975802898406982},
 {'label': 'NEGATIVE', 'score': 0.9767172336578369},
 {'label': 'POSITIVE', 'score': 0.9998652935028076},
 {'label': 'POSITIVE', 'score': 0.9986050724983215},
 {'label': 'NEGATIVE', 'score': 0.9965735673904419},
 {'label': 'POSITIVE', 'score': 0.9998641014099121},
 {'label': 'POSITIVE', 'score': 0.9998828172683716}]

In [29]:
sentiment_df = pd.DataFrame(sentiments)

sentiment_df.rename(columns = {"label": "sentiment_label", "score" : "sentiment_score" }, inplace = True)

data = pd.concat([df, sentiment_df], axis = 1)
print(data[["review_text","sentiment_label","sentiment_score","stars"]])

                                         review_text sentiment_label  \
0  App is fast and easy to use, love the new update!        POSITIVE   
1             Works fine but drains battery quickly.        NEGATIVE   
2    Crashes every time I open it, very frustrating.        NEGATIVE   
3      Reliable for my daily work, a few minor bugs.        POSITIVE   
4      Good features but the interface is confusing.        NEGATIVE   
5  Amazing experience, support team was very help...        POSITIVE   
6     Slow on older devices, but overall acceptable.        POSITIVE   
7    Terrible onboarding, I uninstalled after a day.        NEGATIVE   
8        Solid performance, much better than before.        POSITIVE   
9            Great for collaboration, team loves it.        POSITIVE   

   sentiment_score  stars  
0         0.999312      5  
1         0.994539      3  
2         0.999597      1  
3         0.997580      4  
4         0.976717      3  
5         0.999865      5  
6         0

In [32]:
X_1 = data.drop("stars",axis = 1)
y_1 = data['stars']

In [35]:
Numeric_features_1 = ["user_tenure_days", "sessions_last_7d", "review_length", "sentiment_score"]
Categorical_features_1 = ["platform", "country", "sentiment_label"]

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

preprocessing_1 = ColumnTransformer(transformers = [("Num_scaling", StandardScaler(),Numeric_features_1),
                 ("Cat_scaling", OneHotEncoder(handle_unknown = 'ignore'),Categorical_features_1)])


from sklearn.model_selection import train_test_split

X_train_1,X_test_1,y_train_1,y_test_1 = train_test_split(X_1,y_1,random_state = 42,test_size = 0.3)

from sklearn.ensemble import RandomForestRegressor
steps = [("preprossing_1",preprocessing),
         ("RandomForest", RandomForestRegressor(n_estimators= 300,random_state = 42))]

pipe2 = Pipeline(steps = steps, verbose = True)

In [36]:
pipe2.fit(X_train_1,y_train_1)

Pipeline(steps=[('preprossing_1',
                 ColumnTransformer(transformers=[('Num_scaling',
                                                  StandardScaler(),
                                                  ['user_tenure_days',
                                                   'sessions_last_7d',
                                                   'review_length']),
                                                 ('Cat_scaling',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['platform', 'country'])])),
                ('RandomForest',
                 RandomForestRegressor(n_estimators=300, random_state=42))],
         verbose=True)

In [37]:
y_pred_2 = pipe2.predict(X_test_1)
y_pred_2

array([4.01      , 2.59      , 3.91666667])

In [54]:
from sklearn.metrics import root_mean_squared_error, mean_squared_error, r2_score

rmse_1 = root_mean_squared_error(y_test_1,y_pred_2)
print("RMSE score with Sentiment is : {} ".format(rmse))
r2_Score = r2_score(y_test_1, y_pred_2)
print("r2 score with sentiment is : {} ".format(r2_Score))

RMSE score with Sentiment is : 0.6687827527458902 
r2 score with sentiment is : 0.32909444444444425 


In [41]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

x_embed = model.encode(data['review_text'].tolist(),)

In [42]:
x_embed

array([[-0.03634957, -0.03655104,  0.02748592, ...,  0.02267425,
         0.01843281,  0.11816042],
       [-0.04152446,  0.02039203,  0.02656225, ..., -0.05836979,
         0.08183789,  0.03822821],
       [ 0.0636781 , -0.02726911,  0.00814749, ...,  0.00495198,
         0.00335554,  0.06022098],
       ...,
       [ 0.04307394, -0.04376364, -0.00397504, ...,  0.02898293,
        -0.059315  , -0.01228109],
       [-0.01792821, -0.01020452, -0.04240297, ..., -0.07205258,
         0.03790912,  0.08286872],
       [-0.02654834,  0.06036939, -0.00520933, ...,  0.04732464,
        -0.00880343,  0.05294521]], dtype=float32)

In [43]:
emb_df = pd.DataFrame(x_embed)

In [46]:
X_emb = emb_df
y_emb = data["stars"]

In [47]:
X_train_emb, X_test_emb, y_train_emb, y_test_emb = train_test_split(
    X_emb, y_emb, random_state=42, test_size=0.3
)

In [48]:
rf_emb = RandomForestRegressor(n_estimators=300, random_state=42)
rf_emb.fit(X_train_emb, y_train_emb)

RandomForestRegressor(n_estimators=300, random_state=42)

In [49]:
y_pred_emb = rf_emb.predict(X_test_emb)

In [53]:
print("RMSE:", root_mean_squared_error(y_test_emb, y_pred_emb))
print("r2 Score with embeddings:", r2_score(y_test_emb, y_pred_emb))

RMSE: 1.0723235415572008
r2 Score with embeddings: -0.7248166666666664


In [57]:
print("RMSE score is : {} ".format(rmse))
print("r2 score is : {} ".format(r2_Score))

print("\n")

print("RMSE score with Sentiment is : {} ".format(rmse))
print("r2 score with sentiment is : {} ".format(r2_Score))

print("\n")

print("RMSE:", root_mean_squared_error(y_test_emb, y_pred_emb))
print("r2 Score with embeddings:", r2_score(y_test_emb, y_pred_emb))

RMSE score is : 0.6687827527458902 
r2 score is : 0.32909444444444425 


RMSE score with Sentiment is : 0.6687827527458902 
r2 score with sentiment is : 0.32909444444444425 


RMSE: 1.0723235415572008
r2 Score with embeddings: -0.7248166666666664


Small datasets: Stick to tabular + sentiment (simple, interpretable, robust).

Medium datasets (hundreds): Combine tabular + sentiment + embeddings for hybrid models.

Large datasets (thousands+): choose embeddings or fine-tuned LLMs, since they can generalize semantic meaning better.